# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areebaarain/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
### 1. Ranked actions + reason codes

I use the ranked queue to prioritize pages that are more likely to benefit from review or refresh. The highest-priority pages are those with both relatively high search volume and a longer time since the last update.

Each queue item receives a simple reason code so that a human reviewer can understand why the page was prioritized:

* **STALE_HIGH_VOLUME → REFRESH_NOW:** high search volume and long time since update.
* **STALE → REVIEW_REFRESH:** page appears stale and should be reviewed for a possible refresh.
* **HIGH_VOLUME → REVIEW:** page has meaningful search demand but does not meet the staleness threshold.
* **LOW_OPPORTUNITY → MONITOR:** lower-priority page that does not currently justify immediate refresh work.

The ranking is a prioritization aid, not an automatic instruction to edit a page. A human should review the page before any refresh is approved.
### 1. Ranked actions + reason codes

The queue ranks pages using search volume and days since last update. Pages with both high search demand and greater staleness receive higher priority.

The queue produced 30,000 ranked pages. Of these, 2,541 were marked **REFRESH_NOW**, 6,550 **REVIEW_REFRESH**, 6,599 **REVIEW**, and 14,310 **MONITOR**.

The main reason code for the highest-ranked pages was **STALE_HIGH_VOLUME**, meaning the page has relatively high search volume and has not been updated recently. This gives a human reviewer a clear reason for prioritization.

The queue is a decision-support tool, not an automatic editing system. A human should review the page before approving any refresh action.


In [6]:
!git clone https://github.com/Areebaarain/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [7]:
import pandas as pd
import os

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

File exists: True
Rows: 30000
Columns: 44


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# 1. Ranked actions + reason codes
# ============================================

import pandas as pd
import numpy as np
import os

queue = df.copy()

# --------------------------------------------
# 1. Create percentile scores
# --------------------------------------------

queue["volume_score"] = queue["search_volume"].rank(pct=True)
queue["staleness_score"] = queue["days_since_last_update"].rank(pct=True)

# Combined prioritization score
queue["priority_score"] = (
    0.5 * queue["volume_score"] +
    0.5 * queue["staleness_score"]
)

# --------------------------------------------
# 2. Reason codes
# --------------------------------------------

queue["reason_code"] = np.select(
    [
        (queue["search_volume"] >= queue["search_volume"].quantile(0.75)) &
        (queue["days_since_last_update"] >= queue["days_since_last_update"].quantile(0.75)),

        queue["days_since_last_update"] >= queue["days_since_last_update"].quantile(0.75),

        queue["search_volume"] >= queue["search_volume"].quantile(0.75)
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE",
        "HIGH_VOLUME"
    ],
    default="LOW_OPPORTUNITY"
)

# --------------------------------------------
# 3. Human-readable action
# --------------------------------------------

queue["action"] = queue["reason_code"].map({
    "STALE_HIGH_VOLUME": "REFRESH_NOW",
    "STALE": "REVIEW_REFRESH",
    "HIGH_VOLUME": "REVIEW",
    "LOW_OPPORTUNITY": "MONITOR"
})

# --------------------------------------------
# 4. Rank
# --------------------------------------------

queue = queue.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# --------------------------------------------
# 5. Show top 20
# --------------------------------------------

display(
    queue[
        [
            "rank",
            "content_id",
            "search_volume",
            "days_since_last_update",
            "priority_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

print("\nTotal pages:", len(queue))
print("\nAction counts:")
print(queue["action"].value_counts())


,rank,content_id,search_volume,days_since_last_update,priority_score,reason_code,action
0,1,content_a31e10779c01,3600.0,144,0.992520,STALE_HIGH_VOLUME,REFRESH_NOW
1,2,content_bbca724138f2,1600.0,236,0.991291,STALE_HIGH_VOLUME,REFRESH_NOW
2,3,content_40e140ba2934,720.0,231,0.984434,STALE_HIGH_VOLUME,REFRESH_NOW
3,4,content_24abafed9707,480.0,231,0.978968,STALE_HIGH_VOLUME,REFRESH_NOW
4,5,content_23e958c54c78,480.0,144,0.975976,STALE_HIGH_VOLUME,REFRESH_NOW
5,6,content_c3dd69918c8c,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
6,7,content_29ec1008c834,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
7,8,content_6efb8fa48ebe,210.0,151,0.961461,STALE_HIGH_VOLUME,REFRESH_NOW
8,9,content_0cc405838fc5,210.0,144,0.961002,STALE_HIGH_VOLUME,REFRESH_NOW
9,10,content_17e6b2ba4b08,170.0,144,0.956235,STALE_HIGH_VOLUME,REFRESH_NOW



Total pages: 30000

Action counts:
action
MONITOR           14310
REVIEW             6599
REVIEW_REFRESH     6550
REFRESH_NOW        2541
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
### 2. Intended use and limits

The intended users are **content editors, SEO teams, and content managers** who need to prioritize pages for review and possible refresh.

The playbook is intended to help answer a practical question: **which pages should a human review first for a possible refresh?** It uses search volume and time since the last update to create a ranked priority queue with simple reason codes.

The output is **decision support, not an automatic decision**. A high score does not prove that a page needs a refresh or that refreshing it will improve performance. The analysis shows patterns in the available data, but it does not establish causation.

The queue should not be treated as valid for situations outside the data and evaluation setup used in this project. Changes in search behavior, content type, client mix, or data quality may reduce its usefulness. Human review is required before any content change is approved.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
### 3. Human review + the no-go list

Before taking action, a content editor should review the page and confirm that the priority makes sense. The reviewer should check:

* Whether the page is actually outdated or missing important information.
* Whether the topic and search demand are still relevant.
* Whether the current headline, content, and search intent match.
* Whether important facts, statistics, links, or references need updating.
* Whether the page has other business, editorial, or brand considerations that are not captured by the model.
* Whether a refresh is likely to provide enough value to justify the time and effort.

The following should **never be automated** by this playbook:

* Automatically rewriting or publishing content.
* Automatically deleting or redirecting pages.
* Automatically changing headlines or search strategy.
* Treating a high priority score as proof that a refresh will improve rankings or traffic.
* Making irreversible content or business decisions without human approval.

The model should therefore recommend **what to review first**, while the final content decision remains with a human.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
### 4. Monitoring / retrain triggers

The recommendations should be monitored because search behavior and content performance can change over time.

I would review the playbook periodically and look for these warning signs:

* The ranking of pages changes substantially because search volume or freshness patterns have shifted.
* The distribution of the main features changes compared with the data used to build the queue.
* The model's validation performance declines when checked on newer data.
* Pages marked as high priority are frequently judged by editors to be poor refresh candidates.
* Pages marked as lower priority begin showing stronger evidence that they need attention.
* Search behavior, content types, or the client mix changes substantially.

A retraining or re-evaluation cycle should be triggered when these changes are large or persistent. Before retraining, the data and feature definitions should also be checked for changes or leakage.

Monitoring should therefore focus on **whether the recommendations remain useful to human reviewers**, rather than assuming that the original model will remain valid indefinitely.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
### 5. Exports for the paper

The ranked action queue is exported to `work/outputs/` so that the paper can reuse the same ranked results without manually recreating them.

The exported queue contains the page rank, content ID, search volume, freshness information, priority score, reason code, and recommended action. The queue is regenerated by the notebook rather than committed to Git, while any reusable figures and metric receipts can be kept separately for the paper.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# 5. Exports for the paper
# ============================================

import os

# Create output directory if it does not exist
os.makedirs("work/outputs", exist_ok=True)

# Select useful columns for the paper
queue_export = queue[
    [
        "rank",
        "content_id",
        "search_volume",
        "days_since_last_update",
        "priority_score",
        "reason_code",
        "action"
    ]
].copy()

# Export ranked queue
output_path = "work/outputs/action_playbook_queue.csv"

queue_export.to_csv(output_path, index=False)

print("Queue exported successfully:")
print(output_path)

print("\nRows exported:", len(queue_export))

print("\nFirst 10 rows:")
display(queue_export.head(10))


Queue exported successfully:
work/outputs/action_playbook_queue.csv

Rows exported: 30000

First 10 rows:


,rank,content_id,search_volume,days_since_last_update,priority_score,reason_code,action
0,1,content_a31e10779c01,3600.0,144,0.992520,STALE_HIGH_VOLUME,REFRESH_NOW
1,2,content_bbca724138f2,1600.0,236,0.991291,STALE_HIGH_VOLUME,REFRESH_NOW
2,3,content_40e140ba2934,720.0,231,0.984434,STALE_HIGH_VOLUME,REFRESH_NOW
3,4,content_24abafed9707,480.0,231,0.978968,STALE_HIGH_VOLUME,REFRESH_NOW
4,5,content_23e958c54c78,480.0,144,0.975976,STALE_HIGH_VOLUME,REFRESH_NOW
5,6,content_c3dd69918c8c,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
6,7,content_29ec1008c834,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
7,8,content_6efb8fa48ebe,210.0,151,0.961461,STALE_HIGH_VOLUME,REFRESH_NOW
8,9,content_0cc405838fc5,210.0,144,0.961002,STALE_HIGH_VOLUME,REFRESH_NOW
9,10,content_17e6b2ba4b08,170.0,144,0.956235,STALE_HIGH_VOLUME,REFRESH_NOW


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.